In [1]:
%%script false --no-raise-error
!pip3 install opencv-python numpy matplotlib opendatasets pandas kagglehub

In [2]:
%%script false --no-raise-error
!kaggle datasets download -d adamelkholy/human-ai-artwork
!mkdir data
!unzip human-ai-artwork.zip -d data
!rm *.zip

In [3]:
## Package Imports

In [4]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Hide INFO & WARNING logs

import tensorflow as tf
tf.get_logger().setLevel("ERROR")  # Only show errors



import time
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np 


## Pre-Processing

In [5]:
### Splits and Tuning

In [ ]:
weights = {0: 1.0, 1: 1.0} 
num_classes = 1
batch_size = 32
img_ht = 256
img_wt = 256
data_dir = "./data/data"

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.4,
    subset="training",
    seed=123,
    image_size=(img_ht, img_wt),
    batch_size = batch_size
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.4,
    subset="validation",
    seed=123,
    image_size=(img_ht, img_wt),
    batch_size=batch_size
)

val_batches = int(0.5 * len(val_test_ds))
val_ds = val_test_ds.take(val_batches)
test_ds = val_test_ds.skip(val_batches)



Found 271993 files belonging to 52 classes.
Using 163196 files for training.


I0000 00:00:1739215888.708770  309967 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739215888.709003  309967 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739215888.738433  309967 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739215888.738659  309967 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

Found 271993 files belonging to 52 classes.
Using 108797 files for validation.


### Imbalance Bias

In [7]:
pos = 190549
neg = 81457
in_bias = np.log([pos/neg])
out_bias = tf.keras.initializers.Constant(in_bias)

### Image to Binary Mappings

In [8]:
def image_to_binary(img, label):
    label = tf.cast(label, tf.int32)
    new_label = tf.where(label < 25, 1, 0) # label is number of ai class folders
    return (img, new_label)


train_ds = train_ds.map(image_to_binary)

val_ds = val_ds.map(image_to_binary)

test_ds = test_ds.map(image_to_binary)


In [9]:
## Data Augmentation

In [10]:
def img_augmentation(img, lbl):
    min_scale = 0.5
    max_scale = 2.0
    scale_factor = tf.random.uniform(shape=[], minval=min_scale, maxval=max_scale)
    resized_img = tf.image.resize(img, tf.cast(tf.cast(tf.shape(img)[1:3], tf.float32) * scale_factor, tf.int32))
    rescaled_img = tf.image.resize(resized_img, tf.shape(img)[1:3])
    
    return (rescaled_img, lbl)

train_ds = train_ds.map(img_augmentation)
val_ds = val_ds.map(img_augmentation)
test_ds = test_ds.map(img_augmentation)

def reshape(image, label):
    label = tf.reshape(label, (-1, 1))  # Convert (batch_size,) to (batch_size, 1)
    return image, label

train_ds = train_ds.map(reshape)
val_ds = val_ds.map(reshape)
test_ds = test_ds.map(reshape)

## Model Training

In [11]:
dejaivu_model = tf.keras.Sequential([
  tf.keras.layers.Rescaling(1./255),

  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),

  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),

  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_classes, bias_initializer=out_bias, activation="sigmoid")
])

dejaivu_model.name = "dejAIvu_model"
model_path = "./"

### Metrics

In [12]:
class CustomHistory(tf.keras.callbacks.Callback):
    def __init__(self):
      super(CustomHistory, self).__init__()
      self.losses = []
      self.prcs = []
      self.recalls = []
      self.precisions = []
      self.accuracies = []
      self.mses = []

    """ called upon completion of each batch during training, records all performance metrics """
    def on_train_batch_end(self, batch, logs=None):
      self.losses.append(logs['loss'])
      self.mses.append(logs['MSE'])
      self.accuracies.append(logs['accuracy'])
      self.prcs.append(logs['prc'])
      self.recalls.append(logs['recall'])
      self.precisions.append(logs['precision'])

    """ called upon completion of each batch during testing, records all performance metrics """
    def on_test_batch_end(self, batch, logs=None):
      self.losses.append(logs['loss'])
      self.mses.append(logs['MSE'])
      self.accuracies.append(logs['accuracy'])
      self.prcs.append(logs['prc'])
      self.recalls.append(logs['recall'])
      self.precisions.append(logs['precision'])

    """ return all performance metrics """
    def get_metrics(self):
      return self.losses, self.mses, self.accuracies, self.prcs, self.recalls, self.precisions

metrics = [
      tf.keras.metrics.BinaryAccuracy(name='accuracy'),
      tf.keras.metrics.BinaryCrossentropy(name='cross entropy'), # equiv. to model's loss
      tf.keras.metrics.MeanSquaredError(name='MSE'),
      tf.keras.metrics.TruePositives(name='tp'),
      tf.keras.metrics.FalsePositives(name='fp'),
      tf.keras.metrics.TrueNegatives(name='tn'),
      tf.keras.metrics.FalseNegatives(name='fn'),
      tf.keras.metrics.Precision(name='precision'),
      tf.keras.metrics.Recall(name='recall'),
      tf.keras.metrics.AUC(name='roc', curve='ROC'),             # receiver operating characteristic curve
      tf.keras.metrics.AUC(name='prc', curve='PR'),              # precision-recall curve
]

## Model Training

### Compiling and Evaluating Helpers

In [13]:
class ClipWeights(tf.keras.callbacks.Callback):
    def on_batch_end(self, batch, logs=None):
        """Clips weights to prevent exploding values."""
        clip_value = 1.0  # Adjust if needed
        for layer in self.model.layers:
            if hasattr(layer, "kernel"):
                weights = layer.kernel.numpy()
                clipped_weights = tf.clip_by_value(weights, -clip_value, clip_value)
                layer.kernel.assign(clipped_weights)

def compile_model(model):
  model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=metrics
  )
  return model

def fit_model(model):
  history_callback = CustomHistory()
  model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    verbose=1,
    class_weight=weights,
    callbacks=[history_callback, ClipWeights()]
  )
  return model, history_callback

def evaluate_model_on_test(model):
  eval_metrics = model.evaluate(test_ds)
  return eval_metrics

def save_model(model):
  model_name = model._name
  print("\nSaving " + model_name + ".keras")
  try:
    model.save(model_path + model_name + ".keras")
  except:
    print("Error saving " + model_name + ".keras...")
    return
  print(model_name + ".keras saved successfully.\n")
  
def save_data(data, filename):
  print("Saving data for " + filename)
  try:
    with open(model_path + filename+".txt", 'w') as writefile:
      writefile.write(str(data))
  except:
    print("Error saving data for " + filename)
    return
  print("Data saved.\n")

### Execute Training


In [14]:
from tensorflow.python.client import device_lib
from tensorflow.keras import mixed_precision

policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 5100051331171115008
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 48935665664
locality {
  bus_id: 1
  links {
    link {
      device_id: 1
      type: "StreamExecutor"
      strength: 1
    }
  }
}
incarnation: 2110094376872042451
physical_device_desc: "device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
, name: "/device:GPU:1"
device_type: "GPU"
memory_limit: 48152903680
locality {
  bus_id: 1
  links {
    link {
      type: "StreamExecutor"
      strength: 1
    }
  }
}
incarnation: 15135347619533446627
physical_device_desc: "device: 1, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:41:00.0, compute capability: 8.9"
xla_global_id: 2144165316
]


I0000 00:00:1739215902.699725  309967 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739215902.700001  309967 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739215902.700202  309967 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739215902.700385  309967 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [15]:
for x_batch, y_batch in train_ds.take(1):
    print("Sample input (X):", x_batch.numpy())
    print("Sample labels (Y):", y_batch.numpy())
    break


Sample input (X): [[[[ 96.08334   108.961784   62.96178  ]
   [106.61624   122.03178    75.165306 ]
   [116.94269   133.64983    86.464905 ]
   ...
   [133.53992   139.66785   103.02821  ]
   [126.30022   132.05229    96.0551   ]
   [106.821846  110.333534   78.165855 ]]

  [[108.10927   117.31245    72.59689  ]
   [122.97614   135.29663    88.94445  ]
   [117.85453   133.17339    86.2561   ]
   ...
   [140.00934   146.8924    106.35322  ]
   [144.64685   151.14572   111.45313  ]
   [147.29099   151.54945   116.07922  ]]

  [[111.059074  115.37572    73.72771  ]
   [130.53093   139.9216     91.74498  ]
   [135.32793   149.55524   100.72305  ]
   ...
   [167.78108   173.76616   131.944    ]
   [174.00247   179.56175   139.24973  ]
   [178.5136    181.9221    144.65758  ]]

  ...

  [[208.90851   200.49065   190.38425  ]
   [211.02348   202.60564   192.49924  ]
   [203.4366    195.0406    184.89891  ]
   ...
   [195.49556   186.60887   188.44217  ]
   [188.1607    180.564     180.38033  

In [17]:
from tensorflow.keras.metrics import MeanSquaredError, AUC, Recall, Precision

def train_model_pipeline(lr=0.001):
    start = time.time()

    # Enable Multi-GPU Training
    strategy = tf.distribute.MirroredStrategy()
    print(f"Using {strategy.num_replicas_in_sync} GPU(s) for training.")

    # Define model inside `strategy.scope()` to avoid errors
    with strategy.scope():
        dejaivu_model = tf.keras.Sequential([
            tf.keras.layers.Rescaling(1./255),

            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.MaxPooling2D(),

            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.MaxPooling2D(),

            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dense(num_classes, bias_initializer=out_bias, activation="sigmoid")
        ])
        
        dejaivu_model.name = "dejAIvu_model"

        # Compile the model inside the scope
        dejaivu_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0), 
                              loss="binary_crossentropy", 
                              metrics=["accuracy", Precision(name="precision"), Recall(name="recall"), AUC(name="prc", curve="PR"), MeanSquaredError(name="MSE")])
    


    label_counts = np.array([0, 0])  # Assuming binary labels [0,1]
    for img, lbl in train_ds.unbatch().take(1000):  # Check 1000 samples
        label_counts[int(lbl.numpy())] += 1

    print(f"Class 0: {label_counts[0]} samples, Class 1: {label_counts[1]} samples")
    
    # Train the model inside multi-GPU strategy
    trained_model, history = fit_model(dejaivu_model)

    # Save model and history
    save_data(history.get_metrics(), dejaivu_model.name + "_history")
    save_model(trained_model)

    # Evaluate on test set
    evals = evaluate_model_on_test(trained_model)
    save_data(evals, dejaivu_model.name + "_evals")

    time_taken = time.time() - start
    print(f"Training complete in {round(time_taken/60, 2)} minutes")

    return trained_model, evals, history

# Run training with your custom model inside the correct scope
model, evals, history = train_model_pipeline()


Using 2 GPU(s) for training.
Class 0: 293 samples, Class 1: 707 samples
Epoch 1/3
1383/5100 ━━━━━━━━━━━━━━━━━━━━ 4:15 69ms/step - MSE: nan - accuracy: nan - loss: nan - prc: nan - precision: nan - recall: nan

KeyboardInterrupt: 